[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/rl-lab/blob/main/05_dqn.ipynb)


# 05. Deep Q-Network на CartPole

Цель ноутбука — обучить нейронную аппроксимацию Q-функции для CartPole-v1 с replay buffer и target network.

**Результаты обучения:**
- строить DQN-модель в PyTorch;
- использовать replay buffer для декорреляции данных;
- стабилизировать обучение target network;
- отслеживать решение задачи по средней награде за 100 эпизодов.

## Источник
Lapan M., *Deep Reinforcement Learning Hands-On*, глава 6; Mnih V. et al., 2013 и 2015.


In [ ]:
!pip install -q gymnasium


Подключим Gymnasium, PyTorch и служебные библиотеки. Если в Colab включена GPU, PyTorch автоматически будет использовать CUDA.


In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass
import collections
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


Q-сеть принимает четыре признака состояния CartPole и возвращает оценки ценности для двух действий.


In [ ]:
class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.net(x)


Replay buffer хранит переходы `(s, a, r, s_next, done)`. Для обучения из него выбирается случайный мини-батч.


In [ ]:
ReplayBuffer = collections.deque
buffer = ReplayBuffer(maxlen=10000)

def sample_batch(buffer, batch_size):
    idx = np.random.choice(len(buffer), batch_size, replace=False)
    states, actions, rewards, next_states, dones = zip(*(buffer[i] for i in idx))
    return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
            np.array(next_states), np.array(dones, dtype=np.float32))


Гиперпараметры выбраны так, чтобы ноутбук запускался на CPU, но при наличии GPU обучался быстрее.


In [ ]:
LR = 1e-3
GAMMA = 0.99
BATCH_SIZE = 64
BUFFER_MIN = 1000
EPS_START, EPS_END = 1.0, 0.02
EPS_DECAY_STEPS = 10000
TARGET_SYNC_STEPS = 100
MAX_EPISODES = 1000
SOLVED_REWARD = 195


Создаем policy network и target network. Target network синхронизируется с policy network через фиксированное число глобальных шагов.


In [ ]:
env = gym.make("CartPole-v1")
policy_net = QNet().to(device)
target_net = QNet().to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()
optimizer = optim.Adam(policy_net.parameters(), lr=LR)
episode_rewards, train_log = [], []
global_step = 0


На каждом шаге агент выбирает действие epsilon-greedy, сохраняет переход и, когда буфер наполнен, делает один градиентный шаг DQN.


Вынесем один градиентный шаг в отдельную функцию. Она ничего не делает, пока буфер не накопит минимальный объем переходов.


In [ ]:
def optimize_model():
    if len(buffer) < BUFFER_MIN:
        return None
    states, actions, rewards, next_states, dones = sample_batch(buffer, BATCH_SIZE)
    states_v = torch.tensor(states, dtype=torch.float32, device=device)
    actions_v = torch.tensor(actions, dtype=torch.long, device=device)
    rewards_v = torch.tensor(rewards, dtype=torch.float32, device=device)
    next_v = torch.tensor(next_states, dtype=torch.float32, device=device)
    dones_v = torch.tensor(dones, dtype=torch.float32, device=device)
    q_vals = policy_net(states_v).gather(1, actions_v.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        next_q = target_net(next_v).max(dim=1).values
        target = rewards_v + GAMMA * next_q * (1 - dones_v)
    loss = F.mse_loss(q_vals, target)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    return float(loss.item())


Основной цикл собирает опыт, вызывает `optimize_model()` и периодически синхронизирует target network. Первый эпизод получает seed, а дальше среда сбрасывается без seed, чтобы не повторять траектории.


In [ ]:
for episode in range(1, MAX_EPISODES + 1):
    obs, info = env.reset(seed=SEED if episode == 1 else None)
    done, total_reward = False, 0.0
    while not done:
        eps = max(EPS_END, EPS_START - global_step / EPS_DECAY_STEPS * (EPS_START - EPS_END))
        if np.random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                obs_v = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                action = int(policy_net(obs_v).argmax(dim=1).item())
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        buffer.append((obs, action, reward, next_obs, done))
        obs, total_reward = next_obs, total_reward + reward
        global_step += 1
        optimize_model()
        if global_step % TARGET_SYNC_STEPS == 0:
            target_net.load_state_dict(policy_net.state_dict())
    episode_rewards.append(total_reward)
    rolling_mean = np.mean(episode_rewards[-100:])
    train_log.append((episode, rolling_mean, eps, len(buffer)))
    if episode >= 100 and rolling_mean >= SOLVED_REWARD:
        break
env.close()


Построим кривую средней награды за последние 100 эпизодов. Этот показатель используется как критерий решения CartPole.


In [ ]:
log = pd.DataFrame(train_log, columns=[
    "episode", "rolling_mean_100", "epsilon", "buffer_size"
])
plt.figure(figsize=(8, 4))
plt.plot(log["episode"], log["rolling_mean_100"])
plt.axhline(SOLVED_REWARD, color="red", linestyle="--")
plt.xlabel("Episode")
plt.ylabel("Rolling mean reward")
plt.title("DQN training on CartPole-v1")
plt.grid(alpha=0.3)
plt.show()


Based on Lapan M., *Deep Reinforcement Learning Hands-On*, chapter 6.


In [ ]:
os.makedirs("results", exist_ok=True)
rows = []
for label in [100, 300, 500, 700]:
    part = log[log["episode"] == label]
    if len(part):
        r = part.iloc[0]
        rows.append((label, f"{r['rolling_mean_100']:.2f}", f"{r['epsilon']:.3f}", int(r['buffer_size'])))
    else:
        rows.append((label, "-", "-", "-"))
r = log.iloc[-1]
rows.append(("Финал", f"{r['rolling_mean_100']:.2f}", f"{r['epsilon']:.3f}", int(r['buffer_size'])))
df_results = pd.DataFrame(rows, columns=["Эпизод", "Средняя награда (100 посл.)", "ε", "Размер буфера"])
print(df_results.to_string(index=False))
df_results.to_csv("results/05_dqn_training.csv", index=False)
